# 03 - Data Prep and Model Comparison

This notebook converts EDA decisions into repeatable data prep and compares baseline model families on the same split.


In [9]:
from pathlib import Path
import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, average_precision_score, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

BASE_DIR = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()

candidates = [
    BASE_DIR / "data" / "bank-full.csv",
    BASE_DIR / "data" / "bank.csv",
]

DATA_PATH = next((p for p in candidates if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError("Dataset not found. Tried:\n" + "\n".join(str(p) for p in candidates))

CLEAN_PATH = BASE_DIR / "data" / "bank-clean.csv"
REPORT_PATH = BASE_DIR / "notebooks" / "artifacts" / "03_model_compare_cv.csv"

FEATURES = ["age", "job", "default", "housing", "loan", "marital", "education"]
TARGET = "y"
CAT_FEATURES = ["job", "default", "housing", "loan", "marital", "education"]
NUM_FEATURES = ["age"]

print("Using raw dataset:", DATA_PATH)


Using raw dataset: /home/ian-migwi/Documents/Projects/term_deposit_predictor/data/bank-full.csv


In [10]:
# Load and prep data according to EDA decisions.
df_raw = pd.read_csv(DATA_PATH, sep=";")
df = df_raw[FEATURES + [TARGET]].copy()

for col in CAT_FEATURES + [TARGET]:
    df[col] = df[col].astype(str).str.strip().str.lower()

for col in CAT_FEATURES:
    df[col] = df[col].replace({"": "unknown", "nan": "unknown"}).fillna("unknown")

if df["age"].isna().any():
    df["age"] = df["age"].fillna(df["age"].median())

# Audit duplicates on full raw rows; do not drop unless a data-quality issue is confirmed.
raw_dup_count = int(df_raw.duplicated().sum())
print("Raw exact duplicates:", raw_dup_count)
print("Rows retained for modeling:", len(df))

CLEAN_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(CLEAN_PATH, index=False)
print("Saved cleaned dataset:", CLEAN_PATH)



Raw exact duplicates: 0
Rows retained for modeling: 45211
Saved cleaned dataset: /home/ian-migwi/Documents/Projects/term_deposit_predictor/data/bank-clean.csv


In [11]:
X = df[FEATURES]
y = (df[TARGET] == "yes").astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train size:", X_train.shape, "Test size:", X_test.shape)
print("Target rate (train):", round(y_train.mean(), 4))
print("Target rate (test) :", round(y_test.mean(), 4))


Train size: (36168, 7) Test size: (9043, 7)
Target rate (train): 0.117
Target rate (test) : 0.117


In [12]:
preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), NUM_FEATURES),
        ("cat", OneHotEncoder(handle_unknown="ignore"), CAT_FEATURES),
    ]
)


In [13]:
models = {
    "logistic_regression": LogisticRegression(max_iter=1000, class_weight="balanced"),
    "random_forest": RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1, class_weight="balanced_subsample"),
    "gradient_boosting": GradientBoostingClassifier(random_state=42),
}

scoring = {
    "roc_auc": "roc_auc",
    "pr_auc": "average_precision",
    "f1_yes": "f1_macro",
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
rows = []


In [14]:
def eval_model(name, model, n_jobs=1):
    pipe = Pipeline([
        ("prep", preprocess),
        ("model", model),
    ])
    cv_res = cross_validate(pipe, X_train, y_train, cv=cv, scoring=scoring, n_jobs=n_jobs)
    return {
        "model": name,
        "cv_roc_auc_mean": cv_res["test_roc_auc"].mean(),
        "cv_roc_auc_std": cv_res["test_roc_auc"].std(),
        "cv_pr_auc_mean": cv_res["test_pr_auc"].mean(),
        "cv_pr_auc_std": cv_res["test_pr_auc"].std(),
        "cv_f1_macro_mean": cv_res["test_f1_yes"].mean(),
        "cv_f1_macro_std": cv_res["test_f1_yes"].std(),
    }


In [15]:
rows.append(eval_model("logistic_regression", models["logistic_regression"], n_jobs=1))


In [16]:
rows.append(eval_model("random_forest", models["random_forest"], n_jobs=1))


In [17]:
rows.append(eval_model("gradient_boosting", models["gradient_boosting"], n_jobs=1))


In [18]:
cv_df = pd.DataFrame(rows).sort_values("cv_roc_auc_mean", ascending=False)
display(cv_df)


,model,cv_roc_auc_mean,cv_roc_auc_std,cv_pr_auc_mean,cv_pr_auc_std,cv_f1_macro_mean,cv_f1_macro_std
2,gradient_boosting,0.678341,0.006169,0.253534,0.008185,0.483774,0.003590
0,logistic_regression,0.664121,0.010722,0.216209,0.009352,0.511024,0.005627
1,random_forest,0.624579,0.011265,0.210445,0.007461,0.548922,0.004712


In [19]:
# Fit the top CV model and evaluate on the hold-out test set.
best_name = cv_df.iloc[0]["model"]
best_model = models[best_name]

final_pipe = Pipeline([
    ("prep", preprocess),
    ("model", best_model),
])

final_pipe.fit(X_train, y_train)
proba = final_pipe.predict_proba(X_test)[:, 1]
pred = final_pipe.predict(X_test)

print("Selected model:", best_name)
print("Hold-out ROC-AUC:", round(roc_auc_score(y_test, proba), 4))
print("Hold-out PR-AUC :", round(average_precision_score(y_test, proba), 4))
print("\nClassification report:\n")
print(classification_report(y_test, pred, digits=4))


Selected model: gradient_boosting
Hold-out ROC-AUC: 0.6794
Hold-out PR-AUC : 0.2459

Classification report:

              precision    recall  f1-score   support

           0     0.8836    0.9984    0.9375      7985
           1     0.3810    0.0076    0.0148      1058

    accuracy                         0.8825      9043
   macro avg     0.6323    0.5030    0.4762      9043
weighted avg     0.8248    0.8825    0.8295      9043



In [ ]:
REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
cv_df.to_csv(REPORT_PATH, index=False)
print("Saved model comparison report:", REPORT_PATH)


## Model Selection Criteria

Final model selection is based on:
- validation quality (`ROC-AUC`, `PR-AUC`, `F1`)
- stability (cross-validation standard deviation)
- operational simplicity and maintainability


## Further insights on model selection


  **Why accuracy isn’t enough:** Only ~12% of customers say 'yes' A model can seem accurate by predicting 'no' for almost everyone, while
  still missing most potential subscribers.

  - **Precision**: Of the people we contact, how many say 'yes'?
  - **Recall**: Of all potential “yes” customers, how many did we capture?
  - **PR‑AUC**: Best single‑number summary under class imbalance.

  **Operational takeaway (current model):**
  - The model ranks customers better than random (ROC‑AUC ~0.68).
  - But at the default 0.5 threshold, it misses almost all 'yes' cases.
  - **Business implication:** useful only after threshold tuning or cost‑sensitive optimization.

  **Next step (in a real deployment):**
  Pick a threshold that matches a contact budget or cost‑benefit rule (e.g., “top 10% by score” or “maximize expected profit”), then analyze:
  - expected conversion rate
  - cost per acquisition
  - lift versus random targeting